In [ ]:
"""
Main implementation file for prompt engineering strategies.
COMPLETE THE TODO SECTIONS IN THIS FILE.
"""

from typing import List, Dict, Any
from config import MIN_EXAMPLES, MAX_EXAMPLES
from api_client import call_llm


class PromptStrategy:
    """Base class for prompt strategies."""

    def __init__(self, task_description: str):
        self.task_description = task_description

    def build_prompt(self, input_text: str, **kwargs) -> str:
        """Build a prompt based on the strategy. Override in subclasses."""
        raise NotImplementedError("Subclasses must implement build_prompt()")

    def execute(self, input_text: str, **kwargs) -> str:
        """
        Build prompt and call LLM.

        Args:
            input_text: The input to process
            **kwargs: Additional parameters for prompt building

        Returns:
            LLM response
        """
        prompt = self.build_prompt(input_text, **kwargs)
        return call_llm(prompt)


class ZeroShotStrategy(PromptStrategy):
    """
    Zero-shot prompting strategy that uses clear instructions without examples.

    Task: Complete the build_prompt() method to create effective zero-shot prompts.

    Requirements:
    - Create clear, explicit instructions
    - Include task description
    - Specify expected output format clearly
    - Do NOT include any examples
    """

    def build_prompt(self, input_text: str, **kwargs) -> str:
        """
        Build a zero-shot prompt with clear instructions.

        Args:
            input_text: The main input to process
            **kwargs: Optional parameters (not used in zero-shot)

        Returns:
            A well-structured zero-shot prompt string
        """

        prompt = ""

        # 1. Start with the task description
        prompt = f"{self.task_description}\n\n"
        # 2. Add clear instructions about what to do
        prompt += "Instructions: Follow these steps to classify the sentiment of the text.\n"
        prompt += "1. Read the text carefully.\n"
        prompt += "2. Determine the overall sentiment.\n"
        # 3. Specify the output format (e.g., "Respond with only: Positive, Negative, Mixed, or Neutral")
        prompt += "3. Respond with only: Positive, Negative, Mixed, or Neutral\n"
        # 4. Add the input text at the end
        prompt += f"Text: {input_text}\n"
        prompt += "Classification:"

        return prompt


class FewShotStrategy(PromptStrategy):
    """
    Few-shot prompting strategy that teaches through 3-5 strategic examples.

    Task: Complete the build_prompt() method to create effective few-shot prompts.

    Requirements:
    - Validate that examples list has 3-5 items (raise ValueError if not)
    - Format all examples consistently
    - Clearly separate examples from the actual input
    """

    def __init__(self, task_description: str, examples: List[Dict[str, Any]]):
        super().__init__(task_description)
        self.examples = examples

    def build_prompt(self, input_text: str, **kwargs) -> str:
        """
        Build a few-shot prompt with strategic examples.

        Args:
            input_text: The main input to process
            **kwargs: Optional parameters

        Returns:
            A well-structured few-shot prompt string

        Raises:
            ValueError: If examples list has fewer than 3 or more than 5 items
        """
        # 1. First, validate example count (must be 3-5)
        if not (MIN_EXAMPLES <= len(self.examples)):
            # 2. If invalid, raise ValueError with a clear message
            raise ValueError(F"Too few examples provided. Minimum required is {MIN_EXAMPLES}.")

        if not (MAX_EXAMPLES >= len(self.examples)):
            raise ValueError(F"Too many examples provided. Maximum allowed is {MAX_EXAMPLES}.")


        # 3. Start with task description
        prompt = f"{self.task_description}\n\n"
        # 2. Add "Here are some examples:" or similar
        prompt += "Here are some examples:\n"
        # 5. Format each example consistently (Text: ... / Sentiment: ...)
        for example in self.examples:
            prompt += f"Text: {example['input']}\nSentiment: {example['output']}\n"
        # 6. Add "Now classify:" and the new input
        prompt += f"Now classify:\nText: {input_text}\n"
        # 7. End with "Sentiment:" to prompt the answer
        prompt += "Sentiment:"

        return prompt


class ChainOfThoughtStrategy(PromptStrategy):
    """
    Chain-of-thought prompting strategy that requires step-by-step reasoning.

    Task: Complete the build_prompt() method.

    Requirements:
    - Structure prompts to elicit step-by-step reasoning
    - Include explicit instructions to "think step by step"
    """

    def build_prompt(self, input_text: str, **kwargs) -> str:
        """
        Build a chain-of-thought prompt that elicits step-by-step reasoning.

        Args:
            input_text: The main input to process
            **kwargs: Optional parameters

        Returns:
            A well-structured chain-of-thought prompt string
        """
        prompt = ""

        # 1. Start with the task description
        prompt += f"{self.task_description}\n\n"
        # 2. Add the input text/code to analyze
        prompt += f"Input: {input_text}\n"
        # 3. Explicitly ask for step-by-step reasoning
        prompt += "Work through your analysis step by step:\n"

        return prompt